# KAP-FinQA-TR — Çoklu Seed Tekrarlanabilirlik Analizi
**Notebook 3/4:** Qwen2.5-1.5B, SmolLM2-360M ve TinyLlama-1.1B için seed=0 ve seed=123 deneyleri

Benchmark protokolü gereği her model en az 3 farklı seed (42, 0, 123) ile eğitilmekte,  
ortalama ± standart sapma raporlanmaktadır. Seed=42 sonuçları Notebook 1-2'de üretilmiştir.

**Bu notebook:** seed=0 ve seed=123 deneylerini çalıştırır.

In [ ]:
# HÜCRE 1 — Kurulum
!pip install -q transformers peft bitsandbytes trl datasets \
             accelerate rouge-score nltk huggingface_hub

In [ ]:
# HÜCRE 2 — Google Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
os.chdir('/content/drive/MyDrive/kap_finqa')
for s in ['2_finetune.py', '3_eval_finetuned.py']:
    shutil.copy(f'/content/drive/MyDrive/kap_finqa/{s}', f'/content/{s}')
print('✅ Hazır')
!ls

## Qwen2.5-1.5B — Seed 0 & 123

In [ ]:
# HÜCRE 3 — Qwen2.5-1.5B seed=0 (~2.5 saat)
!python /content/2_finetune.py \
    --model Qwen2.5-1.5B \
    --train_csv /content/drive/MyDrive/kap_finqa/train.csv \
    --val_csv /content/drive/MyDrive/kap_finqa/validation.csv \
    --seed 0

In [ ]:
# HÜCRE 4 — Eval: Qwen seed=0
!python /content/3_eval_finetuned.py \
    --model Qwen2.5-1.5B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv \
    --suffix seed0

In [ ]:
# HÜCRE 5 — Qwen yedekle (seed=0)
import shutil, os
shutil.copy('/content/eval_finetuned_Qwen2.5-1.5B_seed0.json',
            '/content/drive/MyDrive/kap_finqa/eval_finetuned_Qwen2.5-1.5B_seed0.json')
print('✅ seed=0 sonucu kaydedildi')

In [ ]:
# HÜCRE 6 — Qwen2.5-1.5B seed=123 (~2.5 saat)
!python /content/2_finetune.py \
    --model Qwen2.5-1.5B \
    --train_csv /content/drive/MyDrive/kap_finqa/train.csv \
    --val_csv /content/drive/MyDrive/kap_finqa/validation.csv \
    --seed 123

In [ ]:
# HÜCRE 7 — Eval: Qwen seed=123
!python /content/3_eval_finetuned.py \
    --model Qwen2.5-1.5B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv \
    --suffix seed123

In [ ]:
# HÜCRE 8 — Qwen yedekle (seed=123)
import shutil
shutil.copy('/content/eval_finetuned_Qwen2.5-1.5B_seed123.json',
            '/content/drive/MyDrive/kap_finqa/eval_finetuned_Qwen2.5-1.5B_seed123.json')
print('✅ seed=123 sonucu kaydedildi')

## Özet: Mean ± Std Hesaplama

In [ ]:
# HÜCRE 9 — Üç seed sonucunu oku ve istatistik hesapla
import json, numpy as np, os

DRIVE = '/content/drive/MyDrive/kap_finqa'

results = {}
for model in ['Qwen2.5-1.5B', 'SmolLM2-360M', 'TinyLlama-1.1B']:
    scores = []
    for seed_tag in ['', '_seed0', '_seed123']:
        path = f'{DRIVE}/eval_finetuned_{model}{seed_tag}.json'
        if os.path.exists(path):
            with open(path) as f:
                d = json.load(f)
            r1 = d.get('rouge1') or d.get('avg_rouge1') or d.get('mean_rouge1')
            if r1:
                scores.append(float(r1))
    if scores:
        results[model] = {'scores': scores, 'mean': np.mean(scores), 'std': np.std(scores)}
        print(f'{model}: n={len(scores)} | scores={[f"{s:.4f}" for s in scores]} | Mean={np.mean(scores):.4f} ± {np.std(scores):.4f}')
    else:
        print(f'{model}: sonuç bulunamadı — JSON key adını kontrol edin')